## Extracting search events from logs

In [272]:
from driver import assign_topic_number
import glob
from tqdm.notebook import tqdm
import json
import pandas as pd

event_mapping = {
    "TaskStarted": "TS", # Task started
    "querySubmitted": "QS", # submitted user formulated query
    "searchResultGenerated": "SG", # serp generated
    "searchNoResults": "N_SG", # no SERP generated
    "clickedDidYouMeanSuggestion": "QC", # submitted SE corrected query
    "resultExposureStarted": "RV", # SERP result viewing
    "clickedResult": "RC", # SERP result clicked
    "OpenedAnswerBox": "AW", # answer writing
    "TaskEnded": "TE", # Task ended
}

event_sequences = []
relevance_df = pd.read_csv("Data/relevance.csv")

for file_name in tqdm(glob.glob("Data/3rd/*_FULL.log"), total=len(glob.glob("Data/3rd/*_FULL.log"))):
    with open(file_name, 'r') as f:
        log_entries = [json.loads(x) for x in f.readlines()]
    
    for entry in log_entries:
        if entry["type"] in list(event_mapping.keys()):
            uid = int(entry["uid"])
            topic_num = assign_topic_number(uid, entry["actual_topic_number"])
            event_type = entry["type"]
            event_key = event_mapping[event_type]
            event_time = entry["timestamp"]
            if (event_key == "RV") or (event_key == "RC"):
                url = entry["url"]
                relevant_resources = relevance_df.loc[relevance_df["topic_number"]==topic_num]["doc"].tolist()
                if url in relevant_resources:
                    event_key = "R_" + event_key
                else:
                    event_key = "NR_" + event_key
            if (event_key == "SG") and (int(entry["rank"]) > 1):
                continue
            event_sequences.append([uid, topic_num, event_type, event_key, event_time])

for file_name in tqdm(glob.glob("Data/4th/*_FULL.log"), total=len(glob.glob("Data/4th/*_FULL.log"))):
    with open(file_name, 'r') as f:
        log_entries = [json.loads(x) for x in f.readlines()]
    
    for entry in log_entries:
        if entry["type"] in list(event_mapping.keys()):
            uid = int(entry["uid"])
            topic_num = assign_topic_number(uid, entry["actual_topic_number"])
            event_type = entry["type"]
            event_key = event_mapping[event_type]
            event_time = entry["timestamp"]
            if (event_key == "RV") or (event_key == "RC"):
                url = entry["url"]
                relevant_resources = relevance_df.loc[relevance_df["topic_number"]==topic_num]["doc"].tolist()
                if url in relevant_resources:
                    event_key = "R_" + event_key
                else:
                    event_key = "NR_" + event_key
            if (event_key == "SG") and (int(entry["rank"]) > 1):
                continue
            event_sequences.append([uid, topic_num, event_type, event_key, event_time])

event_sequences_df = pd.DataFrame(event_sequences, columns=["uid", "topic_number", "action_name", "action_code", "timestamp"])   
event_sequences_df["timestamp"] = pd.to_datetime(event_sequences_df["timestamp"])   
event_sequences_df = event_sequences_df.sort_values(by=["uid", "topic_number", "timestamp"])
event_sequences_df.to_csv("Data/event_sequences_df_final.csv", index=False)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

In [286]:
df = event_sequences_df.sort_values(
    ["uid", "topic_number", "timestamp"]
).copy()

# Create run IDs for consecutive identical actions
df["run_id"] = (
    df.groupby(["uid", "topic_number"])["action_code"]
      .transform(lambda x: x.ne(x.shift()).cumsum())
)

# Aggregate each run
condensed_df = (
    df.groupby(["uid", "topic_number", "run_id"])
      .agg(
          action_name=("action_name", "first"),
          action_code=("action_code", "first"),
          timestamp=("timestamp", "first"),
          count=("action_code", "size")
      )
      .reset_index()
)

# Append "+" where run length > 1
condensed_df["action_code"] = condensed_df.apply(
    lambda row: f"{row['action_code']}+" if row["count"] > 1 else row["action_code"],
    axis=1
)
condensed_df = condensed_df.drop(columns=["run_id"]).sort_values(by=["uid", "topic_number", "timestamp"])
condensed_df.to_csv("Data/condensed_event_sequences_df_final.csv", index=False)

In [287]:
condensed_df.loc[condensed_df["uid"]==1]

,uid,topic_number,action_name,action_code,timestamp,count
0,1,g3t1,TaskStarted,TS,2026-05-19 15:06:45.708,1
1,1,g3t1,querySubmitted,QS,2026-05-19 15:08:39.525,1
2,1,g3t1,searchResultGenerated,SG,2026-05-19 15:08:41.314,1
3,1,g3t1,resultExposureStarted,R_RV+,2026-05-19 15:08:41.585,2
4,1,g3t1,clickedResult,R_RC,2026-05-19 15:08:58.405,1
5,1,g3t1,resultExposureStarted,R_RV+,2026-05-19 15:09:35.824,2
6,1,g3t1,OpenedAnswerBox,AW,2026-05-19 15:09:38.065,1
7,1,g3t1,TaskEnded,TE,2026-05-19 15:10:10.812,1
8,1,g3t2,TaskStarted,TS,2026-05-19 14:56:45.133,1
9,1,g3t2,querySubmitted,QS+,2026-05-19 14:57:07.106,2


In [288]:
event_sequences_df.loc[event_sequences_df["uid"]==1]

,uid,topic_number,action_name,action_code,timestamp
227,1,g3t1,TaskStarted,TS,2026-05-19 15:06:45.708
228,1,g3t1,querySubmitted,QS,2026-05-19 15:08:39.525
229,1,g3t1,searchResultGenerated,SG,2026-05-19 15:08:41.314
230,1,g3t1,resultExposureStarted,R_RV,2026-05-19 15:08:41.585
231,1,g3t1,resultExposureStarted,R_RV,2026-05-19 15:08:41.588
232,1,g3t1,clickedResult,R_RC,2026-05-19 15:08:58.405
233,1,g3t1,resultExposureStarted,R_RV,2026-05-19 15:09:35.824
234,1,g3t1,resultExposureStarted,R_RV,2026-05-19 15:09:35.828
235,1,g3t1,OpenedAnswerBox,AW,2026-05-19 15:09:38.065
236,1,g3t1,TaskEnded,TE,2026-05-19 15:10:10.812


## Identifying common sequence patterns

In [ ]:
import pandas as pd

event_sequences_df = pd.read_csv("Data/condensed_event_sequences_df_final.csv")
final_answer_df = pd.read_csv("Data/final_answer_df.csv")

In [14]:
event_sequences_df.head()

,uid,topic_number,action_name,action_code,timestamp,count
0,1,g3t1,TaskStarted,TS,2026-05-19 15:06:45.708,1
1,1,g3t1,querySubmitted,QS,2026-05-19 15:08:39.525,1
2,1,g3t1,searchResultGenerated,SG,2026-05-19 15:08:41.314,1
3,1,g3t1,resultExposureStarted,R_RV+,2026-05-19 15:08:41.585,2
4,1,g3t1,clickedResult,R_RC,2026-05-19 15:08:58.405,1


In [15]:
event_sequences_df["uid"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18])

In [16]:
action_key_mapping = enumerate(event_sequences_df["action_code"].unique(), start=1)
action_key_mapping = {code: key for key, code in action_key_mapping}
action_key_mapping

{'TS': 1,
 'QS': 2,
 'SG': 3,
 'R_RV+': 4,
 'R_RC': 5,
 'AW': 6,
 'TE': 7,
 'QS+': 8,
 'R_RV': 9,
 'AW+': 10,
 'NR_RV': 11,
 'NR_RV+': 12,
 'TS+': 13,
 'NR_RC': 14,
 'N_SG': 15,
 'QC': 16}

In [17]:
sequence_id_list = []
event_id_list = []
num_events_list = []
action_key_list = []
search_success_list = []

seq_id = 1
for uid in event_sequences_df["uid"].unique():
    user_df = event_sequences_df[event_sequences_df["uid"]==uid]
    for topic_num in user_df["topic_number"].unique():
        answer_match_ratio = final_answer_df[(final_answer_df["uid"]==uid) & (final_answer_df["topic_number"]==topic_num)]["answer_match_ratio"].values[0]
        # print(f"uid: {uid}, topic_num: {topic_num}, answer_match_ratio: {answer_match_ratio}")
        topic_df = user_df[user_df["topic_number"]==topic_num]
        
        evnt_id = 1
        for index, row in topic_df.iterrows():
            sequence_id_list.append(seq_id)
            event_id_list.append(evnt_id)
            num_events_list.append(1)
            action_key_list.append(action_key_mapping[row["action_code"]])
            if answer_match_ratio == 1.0:
                search_success_list.append(1)
            else:
                search_success_list.append(0)
            
            evnt_id+=1
        
        seq_id+=1

event_sequences_df["sequence_id"] = sequence_id_list
event_sequences_df["event_id"] = event_id_list 
event_sequences_df["num_events"] = num_events_list
event_sequences_df["action_key"] = action_key_list
event_sequences_df["search_success"] = search_success_list

event_sequences_df = event_sequences_df[["sequence_id", "uid", "topic_number", "event_id", "num_events", "action_key", "action_name", "action_code", "timestamp", "search_success"]]

In [18]:
success_action_counts = event_sequences_df.loc[event_sequences_df["search_success"]==1].groupby(["uid"])["topic_number"].value_counts().to_frame().reset_index()
fail_action_counts = event_sequences_df.loc[event_sequences_df["search_success"]==0].groupby(["uid"])["topic_number"].value_counts().to_frame().reset_index()

print(f"Avg number of actions for successful searches: {success_action_counts['count'].mean()}", f"Avg number of actions for failed searches: {fail_action_counts['count'].mean()}")

Avg number of actions for successful searches: 23.02777777777778 Avg number of actions for failed searches: 24.61111111111111


In [19]:
event_sequences_df.head()

,sequence_id,uid,topic_number,event_id,num_events,action_key,action_name,action_code,timestamp,search_success
0,1,1,g3t1,1,1,1,TaskStarted,TS,2026-05-19 15:06:45.708,1
1,1,1,g3t1,2,1,2,querySubmitted,QS,2026-05-19 15:08:39.525,1
2,1,1,g3t1,3,1,3,searchResultGenerated,SG,2026-05-19 15:08:41.314,1
3,1,1,g3t1,4,1,4,resultExposureStarted,R_RV+,2026-05-19 15:08:41.585,1
4,1,1,g3t1,5,1,5,clickedResult,R_RC,2026-05-19 15:08:58.405,1


In [20]:
successful_events = event_sequences_df[event_sequences_df["search_success"]==1]
successful_events.to_csv("Data/successful_sessions.txt", sep=" ", header=False, index=False, columns=["sequence_id", "event_id", "num_events", "action_key"])

failed_events = event_sequences_df[event_sequences_df["search_success"]==0]
failed_events.to_csv("Data/failed_sessions.txt", sep=" ", header=False, index=False, columns=["sequence_id", "event_id", "num_events", "action_key"])

In [21]:
from pycspade.helpers import spade, print_result

# To get raw SPADE output
successful_result = spade(filename='Data/successful_sessions.txt', support=0.2, parse=True, maxgap=1)
failed_result = spade(filename='Data/failed_sessions.txt', support=0.2, parse=True, maxgap=1)


In [22]:
print_result(successful_result)

   Occurs     Accum   Support    Confid      Lift                                                                         Sequence
       35        35 0.9722222       N/A       N/A                                                                              (1) 
       33        33 0.9166667 0.9428571 0.9697959                                                                         (1)->(2) 
       33        33 0.9166667 1.0000000 1.0000000                                                                    (1)->(2)->(3) 
       23        23 0.6388889 0.6969697 0.7603306                                                               (1)->(2)->(3)->(4) 
       10        10 0.2777778 0.4347826 0.5397301                                                          (1)->(2)->(3)->(4)->(5) 
       17        40 0.4722222       N/A       N/A                                                                             (10) 
       16        30 0.4444444 0.9411765 0.9411765                            

In [23]:
print_result(failed_result)

   Occurs     Accum   Support    Confid      Lift                                                                         Sequence
       18        18 1.0000000       N/A       N/A                                                                              (1) 
       15        15 0.8333333 0.8333333 0.9375000                                                                         (1)->(2) 
       12        12 0.6666667 0.8000000 0.8000000                                                                    (1)->(2)->(3) 
        8         8 0.4444444 0.6666667 0.7058824                                                               (1)->(2)->(3)->(4) 
       11        31 0.6111111       N/A       N/A                                                                             (10) 
       10        26 0.5555556 0.9090909 0.9090909                                                                        (10)->(7) 
       13        52 0.7222222       N/A       N/A                            

In [24]:
import pandas as pd

def is_sublist(sub, main):
    n = len(sub)
    if n == 0:
        return True  # An empty list is always a subset
    return any(main[i : i + n] == sub for i in range(len(main) - n + 1))

def save_extracted_sequences(result, action_key_mapping, prune=False):
    reverse_action_key_mapping = {v: k for k, v in action_key_mapping.items()}
    rows = []

    for line in result["seqstrm"].splitlines():
        if not line.strip():
            continue

        pattern, counts = line.split(" -- ")

        occurs, accum = map(int, counts.split())

        decoded = " -> ".join(
            reverse_action_key_mapping[int(x.strip())]
            for x in pattern.split("->")
        )

        rows.append({
            "Sequence": pattern,
            "Decoded_sequence": decoded,
            "Actions": [int(x.strip()) for x in pattern.split("->")],
            "Num_actions": len(decoded.split("->")),
            "Occurs": occurs, # number of sessions the pattern occurs in
            "Accum": accum, # total number of times the pattern occurs across all sessions
            "Support": occurs / result["nsequences"]
        })

    df = pd.DataFrame(rows)
    # df = df.loc[df["Num_actions"] > 1].reset_index(drop=True)
    
    if prune == True:
        closed = []

        for _, row in df.iterrows():
            actions = row["Actions"]
            num_actions = row["Num_actions"]
            support = row["Support"]

            is_closed_flag = True
            candidate_patterns = df.loc[(df["Support"] == support) & (df["Num_actions"] > num_actions)] # all patterns longer than current pattern with same support

            if len(candidate_patterns) > 0:
                for _, candidate in candidate_patterns.iterrows():
                    # print(len(actions), len(candidate["Actions"]))
                    if is_sublist(actions, candidate["Actions"]):
                        is_closed_flag = False

            closed.append(is_closed_flag)


        df["is_closed"] = closed
        df = df.loc[df["is_closed"]==True].drop(columns=["is_closed"]).reset_index(drop=True)
    df = df.sort_values(by=["Support", "Num_actions"], ascending=False).reset_index(drop=True)
            

    # pruning for closed sequences, i.e., retaining sequences which do not have any supersequence with the same support
    # closed = []

    # for _, row in df.iterrows():

    #     seq = row["Sequence"].split("->")
    #     support = row["Support"]

    #     is_closed_flag = True

    #     candidates = df[
    #         (df["Support"] == support)
    #         & (df["Num_actions"] > len(seq))
    #     ]

    #     for candidate in candidates["Sequence"]:
    #         cand_seq = candidate.split("->")

    #         if is_subsequence(seq, cand_seq):
    #             is_closed_flag = False
    #             break

    #     closed.append(is_closed_flag)

    # df["is_closed"] = closed
    # df = (
    #     df[df["is_closed"]]
    #     .drop(columns=["is_closed"])
    #     .reset_index(drop=True)
    # )

    # for _, row in df.iterrows():
    #     pattern = row["Decoded_sequence"]
    #     super_patterns = df[df["Decoded_sequence"].str.contains(pattern) & (df["Support"] == row["Support"]) & (df["Decoded_sequence"] != pattern)]
    #     # print(pattern)
    #     # print(super_patterns)
    #     if len(super_patterns) > 0:
    #         # print(f"{pattern} is not a closed pattern.")
    #         closed.append(False)
    #     else:
    #         # print(f"{pattern} is a closed pattern.")
    #         closed.append(True)

    # df["is_closed"] = closed
    # df = df.loc[df["is_closed"]==True].drop(columns=["is_closed"]).reset_index(drop=True)
    # df = df.sort_values(by=["Support", "num_actions"], ascending=False).reset_index(drop=True)

    return df

In [25]:
successful_df_pruned = save_extracted_sequences(successful_result, action_key_mapping, prune=True)
successful_df_all = save_extracted_sequences(successful_result, action_key_mapping, prune=False)
failed_df_pruned = save_extracted_sequences(failed_result, action_key_mapping, prune=True)
failed_df_all = save_extracted_sequences(failed_result, action_key_mapping, prune=False)

In [26]:
successful_df_pruned.to_csv("Results/successful_patterns_pruned.csv", index=False)
successful_df_all.to_csv("Results/successful_patterns_all.csv", index=False)
successful_df_pruned

,Sequence,Decoded_sequence,Actions,Num_actions,Occurs,Accum,Support
0,3,SG,[3],1,36,36,1.000000
1,7,TE,[7],1,36,36,1.000000
2,2 -> 3,QS -> SG,"[2, 3]",2,35,35,0.972222
3,1,TS,[1],1,35,35,0.972222
4,1 -> 2 -> 3,TS -> QS -> SG,"[1, 2, 3]",3,33,33,0.916667
5,4,R_RV+,[4],1,33,33,0.916667
6,3 -> 4,SG -> R_RV+,"[3, 4]",2,31,31,0.861111
7,2 -> 3 -> 4,QS -> SG -> R_RV+,"[2, 3, 4]",3,30,30,0.833333
8,5,R_RC,[5],1,29,29,0.805556
9,1 -> 2 -> 3 -> 4,TS -> QS -> SG -> R_RV+,"[1, 2, 3, 4]",4,23,23,0.638889


In [27]:
failed_df_pruned.to_csv("Results/failed_patterns_pruned.csv", index=False)
failed_df_all.to_csv("Results/failed_patterns_all.csv", index=False)
failed_df_pruned

,Sequence,Decoded_sequence,Actions,Num_actions,Occurs,Accum,Support
0,1,TS,[1],1,18,18,1.000000
1,3,SG,[3],1,18,18,1.000000
2,7,TE,[7],1,18,18,1.000000
3,4,R_RV+,[4],1,17,17,0.944444
4,3 -> 4,SG -> R_RV+,"[3, 4]",2,16,16,0.888889
5,2 -> 3,QS -> SG,"[2, 3]",2,16,16,0.888889
6,2 -> 3 -> 4,QS -> SG -> R_RV+,"[2, 3, 4]",3,15,15,0.833333
7,1 -> 2,TS -> QS,"[1, 2]",2,15,15,0.833333
8,5,R_RC,[5],1,14,14,0.777778
9,11,NR_RV,[11],1,13,13,0.722222


In [28]:
events = []

for _, row in successful_df_pruned.iterrows():
    sequence = row["Decoded_sequence"]
    success_support = row["Support"]
    success_accum = row["Accum"]
    failed_row = failed_df_all[failed_df_all["Decoded_sequence"]==sequence]
    if len(failed_row) == 0:
        failed_support = 0
        failed_accum = 0
    elif len(failed_row) == 1:
        failed_support = failed_row["Support"].values[0]
        failed_accum = failed_row["Accum"].values[0]
    else:
        print(sequence, failed_row)
    events.append({
        "sequence": sequence,
        "successful_support": success_support,
        "successul_accum": success_accum,
        "failed_support": failed_support,
        "failed_accum": failed_accum
    })

for _, row in failed_df_pruned.iterrows():
    sequence = row["Decoded_sequence"]
    failed_support = row["Support"]
    failed_accum = row["Accum"]
    success_row = successful_df_all[successful_df_all["Decoded_sequence"]==sequence]
    
    if len(success_row) == 0:
        success_support = 0
        success_accum = 0
    elif len(success_row) == 1:
        success_support = success_row["Support"].values[0]
        success_accum = success_row["Accum"].values[0]
    else:
        print(sequence, success_row)
    events.append({
        "sequence": sequence,
        "successful_support": success_support,
        "successul_accum": success_accum,
        "failed_support": failed_support,
        "failed_accum": failed_accum
    })

events_df = pd.DataFrame(events)
events_df.drop_duplicates(subset=["sequence"], inplace=True)
events_df = events_df.sort_values(by=["failed_support"], ascending=False).reset_index(drop=True)
events_df.to_csv("Results/patterns_combined.csv", index=False)

In [29]:
events_df

,sequence,successful_support,successul_accum,failed_support,failed_accum
0,SG,1.000000,36,1.000000,18
1,TE,1.000000,36,1.000000,18
2,TS,0.972222,35,1.000000,18
3,R_RV+,0.916667,33,0.944444,17
4,QS -> SG,0.972222,35,0.888889,16
...,...,...,...,...,...
63,TS -> QS -> SG -> R_RV+ -> R_RC,0.277778,10,0.000000,0
64,R_RV+ -> AW -> TE,0.250000,9,0.000000,0
65,QS -> SG -> R_RV+ -> NR_RV+,0.250000,9,0.000000,0
66,R_RV+ -> R_RC -> AW+,0.222222,8,0.000000,0


## Miscelaneous

In [3]:
# successful_sessions = final_answer_df.loc[final_answer_df["answer_match_ratio"]==1.0]
# failed_sessions = final_answer_df.loc[final_answer_df["answer_match_ratio"]<1.0]

successful_transactions = []
failed_transactions = []

for _, row in final_answer_df.iterrows():
    uid = row["uid"]
    topic_num = row["topic_number"]
    session_actions = event_sequences_df.loc[(event_sequences_df["uid"]==uid) & (event_sequences_df["topic_number"]==topic_num)]
    if row["answer_match_ratio"] == 1.0:
        successful_transactions.append(session_actions["event_key"].tolist())
    else:
        failed_transactions.append(session_actions["event_key"].tolist())

In [4]:
successful_transactions_lengths = [int(len(event_seq)) for event_seq in successful_transactions]
sum(successful_transactions_lengths)/len(successful_transactions_lengths)

41.30555555555556

In [5]:
failed_transactions_lengths = [int(len(event_seq)) for event_seq in failed_transactions]
sum(failed_transactions_lengths)/len(failed_transactions_lengths)

42.388888888888886

In [ ]:
from gsppy.gsp import GSP
from prefixspan import PrefixSpan


# transactions = [
#     ["Bread", "Milk"],
#     ["Bread", "Diaper", "Beer", "Eggs"],
#     ["Milk", "Diaper", "Beer", "Coke"],
# ]

# patterns = GSP(successful_transactions).search(min_support=0.3, verbose=True)
# for level, freq_patterns in enumerate(patterns, start=1):
#     print(f"Level {level}: {freq_patterns}")

ps = PrefixSpan(successful_transactions)
patterns = ps.frequent(0.3*len(successful_transactions), closed=True) # all closed patterns occuring even once
patterns = sorted(patterns, reverse=True)
for sup, patt in patterns:
    print(f"{sup}: {' -> '.join(patt)}")

In [19]:
ps = PrefixSpan(failed_transactions)
patterns = ps.frequent(0.3*len(failed_transactions), closed=True) # all closed patterns occuring even once
patterns = sorted(patterns, reverse=True)
for sup, patt in patterns:
    print(f"{sup}: {' -> '.join(patt)}")

18: TS -> TH -> QS -> SG -> R_RV -> AW -> TE
18: TS -> TH -> QF -> R_RV -> AW -> TE
18: TS -> TH -> QF -> QS -> SG -> R_RV -> AW -> TE
18: TS -> TH -> QF -> AW -> TE
17: TS -> TH -> TH -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> AW -> TE
17: TS -> TH -> QS -> SG -> R_RV -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -